# Clase 9 - Métricas de evaluación

Vamos a ver por qué el accuracy puede mentir, armar matrices de confusión,
calcular precision, recall y F1, y mover el umbral de decisión para elegir
qué error preferimos cometer.

## 1. Detectando fraudes

Simulamos 5000 compras con tarjeta de crédito. De cada compra sabemos el
monto, a cuántos kilómetros de la casa del cliente se hizo, y
cuántas compras más hizo esa tarjeta en la última hora. Solo **100** son
fraudes (el 2%).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=0)

def simular_compras(n, monto_tipico, distancia_tipica, compras_por_hora):
    return pd.DataFrame({
        "monto": rng.lognormal(np.log(monto_tipico), 0.8, n).round(-2),
        "distancia_km": rng.exponential(distancia_tipica, n).round(1),
        "compras_ultima_hora": rng.poisson(compras_por_hora, n),
    })

# Los fraudes suelen ser compras más caras, lejos de casa y muchas seguidas
normales = simular_compras(4900, monto_tipico=15_000, distancia_tipica=8, compras_por_hora=0.5)
fraudes = simular_compras(100, monto_tipico=45_000, distancia_tipica=60, compras_por_hora=2.5)

compras = pd.concat([normales.assign(fraude=0), fraudes.assign(fraude=1)], ignore_index=True)
compras.groupby("fraude").mean().round(1)

In [ ]:
compras["fraude"].value_counts()

Separamos train y test. Usamos `stratify=y` para que los dos conjuntos
tengan la misma proporción de fraudes (si no lo hacemos
podría tocarnos un test casi sin fraudes).

In [ ]:
from sklearn.model_selection import train_test_split

X = compras.drop(columns="fraude")
y = compras["fraude"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)
print("Fraudes en test:", y_test.sum(), "de", len(y_test))

### El primer detector

El modelo más sencillo posible: decir siempre "no es fraude".

In [ ]:
from sklearn.metrics import accuracy_score

pred_sencillo = np.zeros(len(y_test), dtype=int)     # siempre predice 0

print("Accuracy del sencillo:", round(accuracy_score(y_test, pred_sencillo), 3))

98% de accuracy sin mirar los datos. Ahora una regresión
logística de verdad:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

logistica = make_pipeline(StandardScaler(), LogisticRegression())
logistica.fit(X_train, y_train)
pred_logistica = logistica.predict(X_test)

print("Accuracy de la logística:", round(accuracy_score(y_test, pred_logistica), 3))

Mirando solo el accuracy, la logística le gana al sencillo por **un punto**.
¿Vale la pena? Contemos cuántos fraudes encontró cada uno:

In [ ]:
print("Fraudes encontrados por el sencillo: ", ((pred_sencillo == 1) & (y_test == 1)).sum(), "de", y_test.sum())
print("Fraudes encontrados por la logística:", ((pred_logistica == 1) & (y_test == 1)).sum(), "de", y_test.sum())

La diferencia es enorme, pero el accuracy casi no la muestra: como el 98%
de los casos son "no fraude", acertar esos ya da un número altísimo.

## 2. La matriz de confusión

Contemos a mano los cuatro casos posibles para la logística:

In [ ]:
VP = ((pred_logistica == 1) & (y_test == 1)).sum()   # dijo fraude y era fraude
VN = ((pred_logistica == 0) & (y_test == 0)).sum()   # dijo normal y era normal
FP = ((pred_logistica == 1) & (y_test == 0)).sum()   # dijo fraude y era normal (falsa alarma)
FN = ((pred_logistica == 0) & (y_test == 1)).sum()   # dijo normal y era fraude (se escapó)

print(f"VP = {VP}, VN = {VN}, FP = {FP}, FN = {FN}")
print("Total:", VP + VN + FP + FN)

scikit-learn lo calcula con `confusion_matrix`. Importa  el orden! filas
= realidad, columnas = predicción, y la clase 0 va primero:

```
[[VN, FP],
 [FN, VP]]
```

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

confusion_matrix(y_test, pred_logistica)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, titulo in zip(axes, [pred_sencillo, pred_logistica], ["sencillo", "Logística"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=["normal", "fraude"], ax=ax, colorbar=False
    )
    ax.set_title(titulo)
plt.show()

Las dos matrices tienen casi los mismos números en la fila de arriba (las
compras normales). La diferencia está toda en la fila de abajo: los fraudes.

## 3. Precision, recall y F1

- **Precision** = VP / (VP + FP): de las compras que marqué como fraude,
  ¿cuántas lo eran?
- **Recall** = VP / (VP + FN): de los fraudes que había, ¿cuántos encontré?
- **F1**: la media armónica de las dos.

In [ ]:
precision = VP / (VP + FP)
recall = VP / (VP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1:        {f1:.2f}")

Verificamos con scikit-learn:

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision:", round(precision_score(y_test, pred_logistica), 2))
print("Recall:   ", round(recall_score(y_test, pred_logistica), 2))
print("F1:       ", round(f1_score(y_test, pred_logistica), 2))

¿Y el sencillo? Su recall es 0. Su precision es $\frac{0}{0}$: nunca dijo
"fraude", así que no hay nada que medir. scikit-learn avisa con un warning
y devuelve 0; con `zero_division=0` le decimos que ya sabemos y que no
avise.

In [ ]:
print("Recall del sencillo:   ", recall_score(y_test, pred_sencillo))
print("Precision del sencillo:", precision_score(y_test, pred_sencillo, zero_division=0))

`classification_report` muestra todo junto, para cada clase:

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_logistica, target_names=["normal", "fraude"]))

Cómo leerlo:

- La fila **fraude** es la que nos interesa: precision, recall y F1 tomando
  a "fraude" como clase positiva.
- La fila **normal** es lo mismo pero tomando a "normal" como positiva. Da
  todo cerca de 1 porque es la clase fácil.
- `support` es cuántos ejemplos reales hay de cada clase.
- `macro avg` es el promedio simple entre las dos clases. `weighted avg`
  pondera por `support`, así que casi copia a la clase mayoritaria.

## 4. Cáncer de mama: ¿cuál es la clase positiva?

Volvemos al dataset de la sesión 3. En este dataset
`0 = maligno` y `1 = benigno`. scikit-learn toma a la clase `1` como
positiva, así que si no hacemos nada, el recall va a medir cuántos tumores
benignos encontramos, que no es lo que importa.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
print(cancer.target_names)          # ['malignant' 'benign'] -> 0 es maligno

X_cancer = cancer.data
y_cancer = (cancer.target == 0).astype(int)   # ahora 1 = maligno

print("Malignos:", y_cancer.sum(), "| Benignos:", (y_cancer == 0).sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=0, stratify=y_cancer
)

modelo_cancer = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
modelo_cancer.fit(X_train, y_train)
pred = modelo_cancer.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["benigno", "maligno"])
plt.show()
print(classification_report(y_test, pred, target_names=["benigno", "maligno"]))

97% de accuracy, pero hay **2 tumores malignos** que el modelo dijo que
eran benignos. En este problema un falso negativo es muchísimo más grave que
un falso positivo (que solo implica pedir más estudios).

### Moviendo el umbral

`predict` usa un umbral de 0.5 sobre la probabilidad. Con `predict_proba`
podemos usar el umbral que queramos:

In [ ]:
prob_maligno = modelo_cancer.predict_proba(X_test)[:, 1]

filas = []
for umbral in [0.5, 0.3, 0.2, 0.1, 0.05]:
    pred_umbral = (prob_maligno >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_test, pred_umbral).ravel()
    filas.append({
        "umbral": umbral,
        "malignos que se escapan (FN)": fn,
        "falsas alarmas (FP)": fp,
        "precision": round(precision_score(y_test, pred_umbral), 3),
        "recall": round(recall_score(y_test, pred_umbral), 3),
    })

pd.DataFrame(filas)

Bajando el umbral a 0.1 no se escapa ningún tumor maligno, a cambio de 11
falsas alarmas. ¿Es un buen tradeoff? Esa decisión no es del modelo, sino de
los médicos.

Podemos ver el equilibrio completo graficando precision y recall para todos
los umbrales posibles:

In [ ]:
from sklearn.metrics import precision_recall_curve

precisiones, recalls, umbrales = precision_recall_curve(y_test, prob_maligno)

plt.plot(umbrales, precisiones[:-1], label="precision")
plt.plot(umbrales, recalls[:-1], label="recall")
plt.axvline(0.5, color="gray", linestyle="--", label="umbral por defecto")
plt.xlabel("Umbral")
plt.legend()
plt.title("Subir el umbral gana precision y pierde recall")
plt.show()

## 5. Más de dos clases: ¿qué dígitos se confunden?

Con 10 clases la matriz de confusión es de 10×10. Usamos un árbol sobre los
dígitos escritos a mano de la clase pasada. Lo vemos con un modelo que se
equivoca bastante para que haya errores que mirar.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.tree import DecisionTreeClassifier

digitos = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digitos.data, digitos.target, test_size=0.25, random_state=0
)

arbol_digitos = DecisionTreeClassifier(max_depth=8, random_state=0)
arbol_digitos.fit(X_train, y_train)
pred = arbol_digitos.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 3))

fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
plt.show()

El número más grande fuera de la diagonal es el par de dígitos
que más confunde el modelo. ¿Tiene sentido que justo esos dos se parezcan?

In [ ]:
print(classification_report(y_test, pred))

Cada dígito tiene su propia precision y recall. El accuracy nos da un solo
número, pero acá vemos qué dígitos son difíciles para el modelo.

## Ejercicios

Las secciones 4 y 5 reusaron `X_train`, `X_test`, `y_train` e `y_test`.
Antes de arrancar, correr la celda de abajo para volver a separar las
compras.

**Ejercicio 1.** El detector sencillo al revés: un modelo que dice siempre
"fraude". Calcular su matriz de confusión, accuracy, precision, recall y F1
sobre el test de las compras. ¿Qué métrica lo hace saltar más rápido?

**Ejercicio 2.** En el detector de fraudes (`logistica`), armar la misma
tabla de umbrales que hicimos para cáncer de mama, con umbrales `[0.5, 0.3,
0.2, 0.1, 0.05]`. Si el banco quiere encontrar al menos 9 de cada 10 fraudes,
¿qué umbral conviene? ¿Cuántas compras normales bloquearía por cada fraude
que encuentra?

In [ ]:
X = compras.drop(columns="fraude")
y = compras["fraude"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

In [ ]:
# Ejercicio 1: 

In [ ]:
# Ejercicio 2: 